# 문제 2-2 : EnumOutputParser 활용 - 영화 리뷰 감정 분석기

영화 리뷰 텍스트를 입력받아 감정을 "긍정", "부정", "보통" 중 하나로 분류합니다.

- `EnumOutputParser`와 `Enum` 클래스 사용
- 여러 개의 테스트 리뷰로 검증
- 감정 분류 결과를 깔끔하게 출력
- (선택) 분석 실패 시 `OutputFixingParser`로 자동 복구

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("OpenAI Key:", "OK" if OPENAI_API_KEY else "없음", "| Groq Key:", "OK" if GROQ_API_KEY else "없음")

### LLM 제공자 선택

`PROVIDER` 값만 `"groq"` 또는 `"openai"`로 바꾸면 이후 모든 셀의 모델이 바뀝니다.
Groq는 OpenAI 호환 API라서 둘 다 `ChatOpenAI` 클래스를 사용하고 `base_url`만 다릅니다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
from enum import Enum
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException
from langchain_classic.output_parsers import EnumOutputParser, OutputFixingParser

llm = get_llm(temperature=0)

## 2. Enum 정의와 파서

- Enum의 **이름**(`POSITIVE`)은 코드에서 쓰는 값, **값**(`"긍정"`)은 모델이 답해야 하는 문자열입니다.
- 모델 답변이 Enum 값 중 하나와 정확히 일치해야 파싱됩니다. 앞뒤 공백·줄바꿈은 제거되지만 `"긍정입니다."`처럼 군더더기가 붙으면 `OutputParserException`이 발생합니다.
- langchain 1.x에서는 `langchain_classic.output_parsers`에서 import 합니다.

In [ ]:
class Sentiment(Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"


output_parser = EnumOutputParser(enum=Sentiment)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

print(repr(output_parser.parse("긍정")))
print(repr(output_parser.parse(" 보통\n")))

## 3. 프롬프트 + LCEL 체인

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 리뷰 감정 분류 전문가입니다. 리뷰를 읽고 감정을 분류하세요. 설명 없이 값 하나만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | output_parser

## 4. 테스트 리뷰로 검증

결과는 문자열이 아니라 `Sentiment` Enum 멤버이므로 출력할 때 `.value`로 한글 값을 꺼냅니다.

In [ ]:
reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
    "영상미는 최고였는데 결말이 너무 허무해서 아쉬웠어요.",
    "올해 본 영화 중 단연 최고! 두 번 봐도 또 보고 싶네요.",
]

LABEL = {Sentiment.POSITIVE: "😊", Sentiment.NEGATIVE: "😞", Sentiment.NEUTRAL: "😐"}


def print_results(target_chain):
    print("=" * 60)
    for i, review in enumerate(reviews, 1):
        try:
            result = target_chain.invoke({"review": review})
            print(f"{i}. {LABEL[result]} [{result.value}] {review}")
        except OutputParserException as e:
            print(f"{i}. ⚠️ [분석 실패] {review}")
            print(f"   └ {str(e).splitlines()[0]}")
    print("=" * 60)


print_results(chain)

## 5. (선택) OutputFixingParser로 실패 복구

- `OutputFixingParser`는 기존 파서를 감싸서, 파싱에 실패하면 **잘못된 응답 + 오류 내용 + 형식 지침**을 LLM에 다시 보내 수정된 응답을 받아 재파싱합니다.
- 파싱이 **실패할 때만** LLM이 추가로 호출됩니다. 정상 응답에는 비용이 들지 않습니다.
- 수정 후에도 형식에 맞지 않으면 다시 `OutputParserException`이 발생하므로 `try/except`는 그대로 둡니다.

### 5-1. 파서 단독 비교

모델이 지침을 어기고 `"긍정입니다."`라고 답한 상황을 가정합니다.

In [ ]:
fixing_parser = OutputFixingParser.from_llm(llm=llm, parser=output_parser, max_retries=1)

bad_completion = "이 리뷰는 긍정입니다."

try:
    output_parser.parse(bad_completion)
except OutputParserException as e:
    print("1) 기본 파서:", str(e).splitlines()[0])

fixed = fixing_parser.parse(bad_completion)
print(f"2) OutputFixingParser: {fixed} / 값: {fixed.value}")

> **주의**: `OutputFixingParser`는 원래 리뷰를 보지 못하고 **잘못된 응답 + 형식 지침**만 보고 고칩니다.
> 위처럼 응답 안에 정답(`긍정`)이 들어 있으면 대부분 제대로 고치지만, 모델에 따라 `보통` 같은 다른 값으로 바꿀 수도 있습니다.
> 즉 **형식은 보장하지만 의미까지 보장하지는 않습니다.** 원래 입력까지 다시 보여주고 재시도하려면 `RetryWithErrorOutputParser`를 사용합니다.

### 5-2. 체인에 연결

In [ ]:
fixing_chain = prompt | llm | fixing_parser

print_results(fixing_chain)